In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# RealMLP + TabArena — Piloto Fragmentado no Kaggle

Este notebook executa uma validação piloto mínima do projeto RealMLP + TabArena no Kaggle.

O pacote do projeto foi anexado como Kaggle Dataset e aparece já descompactado em `/kaggle/input`.

Objetivos desta execução:
- localizar o projeto anexado em `/kaggle/input`;
- copiar o projeto para `/kaggle/working/project`, pois `/kaggle/input` é somente leitura;
- preparar o diretório `/kaggle/working/results`;
- validar estrutura do projeto;
- validar imports principais;
- confirmar a lista de 30 datasets recomendados;
- inspecionar os runners disponíveis;
- executar somente um fragmento pequeno por `task_id`;
- salvar os resultados em `/kaggle/working/results`.

Restrições desta etapa:
- não executar todos os datasets;
- não executar AutoGluon Extreme;
- não rodar notebooks completos pesados;
- não iniciar execução em lote;
- não salvar resultados finais fora de `/kaggle/working/results`.

Este notebook é apenas um piloto operacional para validar se o pacote Kaggle está funcional.

In [4]:
# Esta célula lista recursivamente o conteúdo de /kaggle/input.
# Objetivo:
# - confirmar como o Dataset foi montado pelo Kaggle;
# - identificar o caminho real do projeto anexado;
# - verificar se o projeto já veio descompactado.

from pathlib import Path

input_root = Path("/kaggle/input")

print("Existe /kaggle/input?", input_root.exists())
print("\nConteúdo recursivo de /kaggle/input:")

all_paths = sorted(input_root.rglob("*"))

if not all_paths:
    print("Nenhum arquivo encontrado dentro de /kaggle/input.")
else:
    for path in all_paths:
        kind = "DIR " if path.is_dir() else "FILE"
        size = path.stat().st_size if path.is_file() else "-"
        print(f"{kind} | {path} | size={size}")

Existe /kaggle/input? True

Conteúdo recursivo de /kaggle/input:
DIR  | /kaggle/input/datasets | size=-
DIR  | /kaggle/input/datasets/kischenah | size=-
DIR  | /kaggle/input/datasets/kischenah/realmlp-tabarena-kaggle-package | size=-
FILE | /kaggle/input/datasets/kischenah/realmlp-tabarena-kaggle-package/README.md | size=6910
DIR  | /kaggle/input/datasets/kischenah/realmlp-tabarena-kaggle-package/data | size=-
FILE | /kaggle/input/datasets/kischenah/realmlp-tabarena-kaggle-package/data/__init__.py | size=0
FILE | /kaggle/input/datasets/kischenah/realmlp-tabarena-kaggle-package/data/load_tabarena.py | size=6337
DIR  | /kaggle/input/datasets/kischenah/realmlp-tabarena-kaggle-package/docs | size=-
FILE | /kaggle/input/datasets/kischenah/realmlp-tabarena-kaggle-package/docs/HISTORY.md | size=17898
FILE | /kaggle/input/datasets/kischenah/realmlp-tabarena-kaggle-package/docs/INFRAESTRUTURA.md | size=8840
FILE | /kaggle/input/datasets/kischenah/realmlp-tabarena-kaggle-package/docs/KAGGLE_CHEC

In [5]:
# Esta célula define o caminho do projeto dentro de /kaggle/input.
# Como o Kaggle já montou o pacote descompactado, não precisamos procurar nem extrair ZIP.

from pathlib import Path

input_project_dir = Path("/kaggle/input/datasets/kischenah/realmlp-tabarena-kaggle-package")

assert input_project_dir.exists(), f"Projeto não encontrado em: {input_project_dir}"
assert (input_project_dir / "src").exists(), "Diretório src não encontrado."
assert (input_project_dir / "data").exists(), "Diretório data não encontrado."
assert (input_project_dir / "notebooks").exists(), "Diretório notebooks não encontrado."
assert (input_project_dir / "pyproject.toml").exists(), "pyproject.toml não encontrado."

print("Projeto encontrado em:")
print(input_project_dir)

Projeto encontrado em:
/kaggle/input/datasets/kischenah/realmlp-tabarena-kaggle-package


In [6]:
# Esta célula copia o projeto de /kaggle/input para /kaggle/working/project.
#
# Motivo:
# - /kaggle/input é somente leitura;
# - /kaggle/working é gravável;
# - os scripts podem precisar criar arquivos temporários;
# - os resultados devem ser preservados em /kaggle/working/results.

import shutil
from pathlib import Path

project_dir = Path("/kaggle/working/project")
results_dir = Path("/kaggle/working/results")

if project_dir.exists():
    shutil.rmtree(project_dir)

shutil.copytree(input_project_dir, project_dir)

results_dir.mkdir(parents=True, exist_ok=True)

print("Projeto copiado para:")
print(project_dir)

print("\nDiretório de resultados criado/confirmado:")
print(results_dir)

Projeto copiado para:
/kaggle/working/project

Diretório de resultados criado/confirmado:
/kaggle/working/results


In [7]:
# Esta célula confirma a estrutura principal do projeto já copiado para /kaggle/working/project.
# Ela serve como verificação antes de instalar dependências ou executar scripts.

from pathlib import Path

print("Conteúdo de /kaggle/working/project:")

for path in sorted(project_dir.iterdir()):
    print("-", path.name)

required_paths = [
    project_dir / "src",
    project_dir / "data",
    project_dir / "notebooks",
    project_dir / "docs",
    project_dir / "pyproject.toml",
]

for path in required_paths:
    assert path.exists(), f"Item obrigatório ausente: {path}"

print("\nEstrutura mínima validada.")

Conteúdo de /kaggle/working/project:
- README.md
- data
- docs
- notebooks
- pyproject.toml
- src

Estrutura mínima validada.


In [8]:
# Esta célula define /kaggle/working/project como diretório atual.
# A partir daqui, os comandos são executados dentro da raiz do projeto.

import os
from pathlib import Path

os.chdir(project_dir)

print("Diretório atual:")
print(Path.cwd())

Diretório atual:
/kaggle/working/project


In [9]:
# Esta célula verifica quais arquivos de dependência existem no pacote.
# No pacote atual foi identificado pyproject.toml, então a instalação deve usar o projeto local.

from pathlib import Path

dependency_files = [
    "requirements.txt",
    "requirements-dev.txt",
    "pyproject.toml",
]

for filename in dependency_files:
    path = Path(filename)
    print(filename, "=>", "existe" if path.exists() else "não existe")

requirements.txt => não existe
requirements-dev.txt => não existe
pyproject.toml => existe


In [10]:
# Esta célula instala o projeto local em modo editável.
# Objetivo:
# - permitir imports de src, data e módulos internos;
# - instalar dependências declaradas no pyproject.toml;
# - preparar o ambiente para um piloto pequeno.

import sys
import subprocess

cmd = [
    sys.executable,
    "-m",
    "pip",
    "install",
    "-e",
    ".",
]

print("Executando:")
print(" ".join(cmd))

subprocess.check_call(cmd)

print("\nInstalação concluída.")

Executando:
/usr/bin/python3 -m pip install -e .
Obtaining file:///kaggle/working/project
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
datasets 4.8.5 requires pyarrow>=21.0.0, but you have pyarrow 20.0.0 which is incompatible.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
jupyterlab-lsp 3.10.2 requires jupyterlab<4.0.0a0,>=3.1.0, but you have jupyterlab 4.5.8 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
dask-cuda 26.2.0 requires cuda-core=


Instalação concluída.


In [11]:
# Esta célula valida os imports principais do projeto.
# Também confirma que a lista RECOMMENDED_TASK_IDS mantém os 30 datasets definidos no projeto.

from data.load_tabarena import RECOMMENDED_TASK_IDS

print("Total de task_ids recomendados:", len(RECOMMENDED_TASK_IDS))
print("Task IDs recomendados:")
print(RECOMMENDED_TASK_IDS)

assert len(RECOMMENDED_TASK_IDS) == 30, "A lista RECOMMENDED_TASK_IDS não tem 30 datasets."

print("\nImports e lista de datasets validados.")

Total de task_ids recomendados: 30
Task IDs recomendados:
[363621, 363629, 363614, 363626, 363685, 363696, 363707, 363671, 363711, 363682, 363684, 363674, 363700, 363702, 363620, 363677, 363704, 363623, 363694, 363706, 363619, 363676, 363712, 363632, 363691, 363681, 363679, 363627, 363613, 363699]

Imports e lista de datasets validados.


In [12]:
# Esta célula escolhe apenas um dataset para o piloto fragmentado.
# A execução piloto deve ser pequena e controlada.
#
# Não altere para loop nos 30 datasets nesta etapa.

from data.load_tabarena import RECOMMENDED_TASK_IDS

pilot_task_id = RECOMMENDED_TASK_IDS[0]

print("Task ID piloto selecionado:", pilot_task_id)

Task ID piloto selecionado: 363621


In [13]:
# Esta célula mostra os parâmetros aceitos pelos runners principais.
# Objetivo:
# - descobrir as flags corretas;
# - evitar executar comando errado;
# - confirmar suporte a execução fragmentada por task_id;
# - confirmar suporte a output-dir/checkpoint/resume, se disponível.

import subprocess
import sys

candidate_scripts = [
    "src/pipeline/run_all.py",
    "src/pipeline/run_autogluon.py",
]

for script in candidate_scripts:
    print("\n" + "=" * 100)
    print("Ajuda de:", script)
    print("=" * 100)

    result = subprocess.run(
        [sys.executable, script, "--help"],
        text=True,
        capture_output=True,
    )

    print("STDOUT:")
    print(result.stdout)

    if result.stderr:
        print("STDERR:")
        print(result.stderr)

    print("Return code:", result.returncode)


Ajuda de: src/pipeline/run_all.py
STDOUT:
usage: run_all.py [-h] [--seed SEED] [--train-output TRAIN_OUTPUT]
                  [--test-output TEST_OUTPUT] [--output OUTPUT]
                  [--task-ids [TASK_IDS ...]] [--include-group-model]
                  [--include-hpo] [--hpo-time-limit HPO_TIME_LIMIT]

options:
  -h, --help            show this help message and exit
  --seed SEED
  --train-output TRAIN_OUTPUT
                        caminho do CSV de saída com métricas no conjunto de
                        treinamento
  --test-output TEST_OUTPUT
                        caminho do CSV de saída com métricas no conjunto de
                        teste
  --output OUTPUT       compatibilidade legada: se informado, também salva as
                        métricas de teste neste caminho
  --task-ids [TASK_IDS ...]
                        opcional: lista de task IDs do OpenML; se omitido, usa
                        RECOMMENDED_TASK_IDS
  --include-group-model
                      

In [14]:
# Esta célula executa um piloto mínimo com apenas um dataset.
#
# O runner usado é src/pipeline/run_all.py.
# Nesta primeira validação, NÃO usamos:
# - --include-group-model
# - --include-hpo
#
# Objetivo:
# - validar execução fragmentada por task_id;
# - validar geração separada de métricas de treino e teste;
# - salvar tudo em /kaggle/working/results;
# - evitar execução pesada.

import subprocess
import sys
from pathlib import Path

results_dir = Path("/kaggle/working/results")
results_dir.mkdir(parents=True, exist_ok=True)

pilot_train_output = results_dir / f"pilot_run_all_train_task_{pilot_task_id}.csv"
pilot_test_output = results_dir / f"pilot_run_all_test_task_{pilot_task_id}.csv"
pilot_legacy_output = results_dir / f"pilot_run_all_legacy_test_task_{pilot_task_id}.csv"
pilot_log_output = results_dir / f"pilot_run_all_task_{pilot_task_id}.log"

cmd = [
    sys.executable,
    "src/pipeline/run_all.py",
    "--seed",
    "42",
    "--task-ids",
    str(pilot_task_id),
    "--train-output",
    str(pilot_train_output),
    "--test-output",
    str(pilot_test_output),
    "--output",
    str(pilot_legacy_output),
]

print("Comando executado:")
print(" ".join(cmd))

result = subprocess.run(
    cmd,
    text=True,
    capture_output=True,
)

log_text = (
    "COMANDO:\n"
    + " ".join(cmd)
    + "\n\nSTDOUT:\n"
    + result.stdout
    + "\n\nSTDERR:\n"
    + result.stderr
    + f"\n\nRETURN_CODE: {result.returncode}\n"
)

pilot_log_output.write_text(log_text, encoding="utf-8")

print("STDOUT:")
print(result.stdout)

print("STDERR:")
print(result.stderr)

print("Return code:", result.returncode)
print("Log salvo em:", pilot_log_output)

assert result.returncode == 0, "Execução piloto falhou. Verifique STDOUT/STDERR acima."

Comando executado:
/usr/bin/python3 src/pipeline/run_all.py --seed 42 --task-ids 363621 --train-output /kaggle/working/results/pilot_run_all_train_task_363621.csv --test-output /kaggle/working/results/pilot_run_all_test_task_363621.csv --output /kaggle/working/results/pilot_run_all_legacy_test_task_363621.csv
STDOUT:

[1/1  100%] blood-transfusion-service-center  n=748  cls=2  reg=small
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM]

In [15]:
# Esta célula lista os arquivos gerados em /kaggle/working/results.
# Ela confirma se o piloto salvou os artefatos no local correto.

from pathlib import Path

results_dir = Path("/kaggle/working/results")

print("Arquivos em /kaggle/working/results:")

for path in sorted(results_dir.rglob("*")):
    if path.is_file():
        print("-", path, "|", path.stat().st_size, "bytes")

Arquivos em /kaggle/working/results:
- /kaggle/working/results/pilot_run_all_legacy_test_task_363621.csv | 642 bytes
- /kaggle/working/results/pilot_run_all_task_363621.log | 29860 bytes
- /kaggle/working/results/pilot_run_all_test_task_363621.csv | 642 bytes
- /kaggle/working/results/pilot_run_all_train_task_363621.csv | 649 bytes


In [16]:
# Esta célula carrega os CSVs de treino e teste gerados no piloto.
# Objetivo:
# - confirmar que os arquivos não estão vazios;
# - verificar colunas;
# - conferir quais modelos foram executados;
# - confirmar que os resultados pertencem somente ao task_id piloto.

import pandas as pd

train_df = pd.read_csv(pilot_train_output)
test_df = pd.read_csv(pilot_test_output)

print("TRAIN shape:", train_df.shape)
display(train_df)

print("\nTEST shape:", test_df.shape)
display(test_df)

print("\nTask IDs no treino:", sorted(train_df["task_id"].unique()) if "task_id" in train_df.columns else "coluna task_id ausente")
print("Task IDs no teste:", sorted(test_df["task_id"].unique()) if "task_id" in test_df.columns else "coluna task_id ausente")

if "model" in train_df.columns:
    print("\nModelos no treino:", sorted(train_df["model"].unique()))

if "model" in test_df.columns:
    print("Modelos no teste:", sorted(test_df["model"].unique()))

TRAIN shape: (3, 10)


,task_id,dataset,model,auc_ovo,accuracy,g_mean,cross_entropy,fit_time_s,predict_time_s,total_time_s
0,363621,blood-transfusion-service-center,lightgbm,0.774992,0.791587,0.451549,0.464807,3.644254,0.011848,3.656103
1,363621,blood-transfusion-service-center,xgboost,0.864207,0.843212,0.628602,0.401188,1.326030,0.012759,1.338789
2,363621,blood-transfusion-service-center,catboost,0.823733,0.829828,0.649088,0.477918,2.638872,0.012290,2.651162



TEST shape: (3, 10)


,task_id,dataset,model,auc_ovo,accuracy,g_mean,cross_entropy,fit_time_s,predict_time_s,total_time_s
0,363621,blood-transfusion-service-center,lightgbm,0.753790,0.768889,0.378087,0.484037,3.644254,0.010538,3.654792
1,363621,blood-transfusion-service-center,xgboost,0.751516,0.768889,0.457413,0.487717,1.326030,0.010908,1.336938
2,363621,blood-transfusion-service-center,catboost,0.773446,0.777778,0.553211,0.514314,2.638872,0.008289,2.647161



Task IDs no treino: [np.int64(363621)]
Task IDs no teste: [np.int64(363621)]

Modelos no treino: ['catboost', 'lightgbm', 'xgboost']
Modelos no teste: ['catboost', 'lightgbm', 'xgboost']


In [17]:
# Esta célula valida se o piloto respeitou as restrições da etapa.
#
# Verificações:
# - há métricas de treino;
# - há métricas de teste;
# - somente um task_id foi executado;
# - o task_id executado é o task_id piloto;
# - AutoGluon Extreme não foi executado;
# - execução em lote dos 30 datasets não aconteceu.

assert len(train_df) > 0, "CSV de treino está vazio."
assert len(test_df) > 0, "CSV de teste está vazio."

assert "task_id" in train_df.columns, "CSV de treino não tem coluna task_id."
assert "task_id" in test_df.columns, "CSV de teste não tem coluna task_id."

train_task_ids = set(train_df["task_id"].astype(int).unique())
test_task_ids = set(test_df["task_id"].astype(int).unique())

assert train_task_ids == {int(pilot_task_id)}, f"Treino executou task_ids inesperados: {train_task_ids}"
assert test_task_ids == {int(pilot_task_id)}, f"Teste executou task_ids inesperados: {test_task_ids}"

combined_text = " ".join(
    list(train_df.astype(str).values.flatten()) +
    list(test_df.astype(str).values.flatten())
).lower()

assert "extreme" not in combined_text, "AutoGluon Extreme apareceu nos resultados, o que não era permitido nesta etapa."

print("Validações do piloto concluídas com sucesso.")
print("Task ID executado:", pilot_task_id)
print("Linhas treino:", len(train_df))
print("Linhas teste:", len(test_df))

Validações do piloto concluídas com sucesso.
Task ID executado: 363621
Linhas treino: 3
Linhas teste: 3


In [18]:
# Esta célula cria um manifesto JSON da execução piloto.
# O manifesto registra explicitamente que esta execução foi fragmentada e não pesada.

from pathlib import Path
from datetime import datetime, timezone
import json

manifest = {
    "project": "RealMLP + TabArena",
    "execution_type": "pilot_fragmented_run_all",
    "runner": "src/pipeline/run_all.py",
    "seed": 42,
    "task_id": int(pilot_task_id),
    "train_output": str(pilot_train_output),
    "test_output": str(pilot_test_output),
    "legacy_output": str(pilot_legacy_output),
    "log_output": str(pilot_log_output),
    "results_dir": "/kaggle/working/results",
    "ran_all_datasets": False,
    "ran_autogluon_default": False,
    "ran_autogluon_extreme": False,
    "included_group_model": False,
    "included_hpo": False,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}

manifest_path = results_dir / f"pilot_manifest_task_{pilot_task_id}.json"

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print("Manifesto salvo em:")
print(manifest_path)

Manifesto salvo em:
/kaggle/working/results/pilot_manifest_task_363621.json


In [19]:
# Esta célula compacta os resultados do piloto em um arquivo ZIP.
# O arquivo fica em /kaggle/working para facilitar download pelo painel do Kaggle.

import shutil
from pathlib import Path

archive_base = Path("/kaggle/working/realmlp_tabarena_pilot_results")
archive_path = shutil.make_archive(
    base_name=str(archive_base),
    format="zip",
    root_dir=str(results_dir),
)

print("Arquivo ZIP criado:")
print(archive_path)

Arquivo ZIP criado:
/kaggle/working/realmlp_tabarena_pilot_results.zip
